# 02 Synthetic Event Backtest

I use this notebook to build a small event-driven backtest on synthetic prediction-market data.

The focus is the mechanics: a walk-forward table, entry and exit bookkeeping, transaction costs, and basic trade summaries without private market data or private strategy rules.

## Step 0: Imports And Paths

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

from data.synthetic_data import make_synthetic_market_data, make_synthetic_event_windows
from data.timestamp_alignment import align_prediction_to_underlying
from features.rolling_features import add_rolling_zscore

## Step 1: Build Synthetic Inputs

The event windows are artificial. They give the table a simple structure for event-driven entry and exit logic.

In [ ]:
underlying, prediction = make_synthetic_market_data(
    seed=21,
    periods=480,
)

aligned = align_prediction_to_underlying(
    underlying,
    prediction,
    max_stale_minutes=20,
).sort_values("timestamp_utc").reset_index(drop=True)

event_windows = make_synthetic_event_windows(
    aligned["timestamp_utc"],
    event_every=90,
    window_minutes=25,
)

display(event_windows)
display(aligned.head())

## Step 2: Mark Event Windows

I keep the loop explicit: each event window creates a boolean mask on the main table.

In [ ]:
event_table = aligned.copy()
event_table["in_event_window"] = False
event_table["event_id"] = pd.NA

for _, event in event_windows.iterrows():
    event_mask = (
        (event_table["timestamp_utc"] >= event["event_start_utc"])
        & (event_table["timestamp_utc"] <= event["event_end_utc"])
    )
    event_table.loc[event_mask, "in_event_window"] = True
    event_table.loc[event_mask, "event_id"] = event["event_id"]

print("event-window rows:", event_table["in_event_window"].sum())
display(event_table[event_table["in_event_window"]].head(10))

## Step 3: Create Placeholder Entry Conditions

I use a rolling z-score as a public-safe stand-in for signal research. The condition is active only inside synthetic event windows and only when a current prediction price exists.

In [ ]:
FEATURE_WINDOW = 30
ENTRY_Z = 1.10

working_table = add_rolling_zscore(
    event_table,
    column="underlying_return",
    window=FEATURE_WINDOW,
    min_periods=FEATURE_WINDOW,
)

z_col = f"underlying_return_zscore_{FEATURE_WINDOW}"

long_condition = (
    working_table["in_event_window"]
    & working_table["prediction_price"].notna()
    & (working_table[z_col] >= ENTRY_Z)
)

short_condition = (
    working_table["in_event_window"]
    & working_table["prediction_price"].notna()
    & (working_table[z_col] <= -ENTRY_Z)
)

working_table["entry_signal"] = 0
working_table.loc[long_condition, "entry_signal"] = 1
working_table.loc[short_condition, "entry_signal"] = -1

display(working_table[[
    "timestamp_utc",
    "event_id",
    "prediction_price",
    "prediction_stale_minutes",
    z_col,
    "entry_signal",
]].query("entry_signal != 0").head(20))

## Step 4: Walk The Table One Row At A Time

The loop allows one open position at a time. Exits happen after a fixed toy holding period, at an opposite placeholder signal, or at the final available row.

In [ ]:
RETURN_HORIZON_MINS = 20
COST_PER_SIDE = 0.002
TRADE_SIZE = 1.0

trade_log = []
position = 0
entry_time = None
entry_price = None
entry_side = None
entry_cost = None

last_index = working_table.index[-1]

for idx, row in working_table.iterrows():
    price_available = pd.notna(row["prediction_price"])

    if position == 0 and price_available and row["entry_signal"] != 0:
        position = int(row["entry_signal"])
        entry_time = row["timestamp_utc"]
        entry_price = row["prediction_price"]
        entry_side = "long" if position == 1 else "short"
        entry_cost = COST_PER_SIDE * TRADE_SIZE

    if position != 0 and price_available:
        held_minutes = (row["timestamp_utc"] - entry_time).total_seconds() / 60
        opposite_signal = row["entry_signal"] == -position
        time_exit = held_minutes >= RETURN_HORIZON_MINS
        final_row_exit = idx == last_index

        if time_exit or opposite_signal or final_row_exit:
            exit_price = row["prediction_price"]
            raw_pnl = position * (exit_price - entry_price) * TRADE_SIZE
            total_cost = entry_cost + COST_PER_SIDE * TRADE_SIZE
            net_pnl = raw_pnl - total_cost

            trade_log.append({
                "entry_time": entry_time,
                "exit_time": row["timestamp_utc"],
                "entry_side": entry_side,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "held_minutes": held_minutes,
                "raw_pnl": raw_pnl,
                "total_cost": total_cost,
                "net_pnl": net_pnl,
                "exit_reason": "time" if time_exit else "opposite_signal" if opposite_signal else "final_row",
            })

            position = 0
            entry_time = None
            entry_price = None
            entry_side = None
            entry_cost = None

trade_log_df = pd.DataFrame(trade_log)
display(trade_log_df)

## Step 5: Trade Summary

I separate gross movement from transaction-cost-adjusted results.

In [ ]:
if len(trade_log_df) == 0:
    trade_summary_df = pd.DataFrame([{"trades": 0}])
else:
    trade_summary = {
        "trades": len(trade_log_df),
        "long_pct": (trade_log_df["entry_side"] == "long").mean(),
        "total_raw_pnl": trade_log_df["raw_pnl"].sum(),
        "total_net_pnl": trade_log_df["net_pnl"].sum(),
        "average_net_pnl": trade_log_df["net_pnl"].mean(),
        "best_trade": trade_log_df["net_pnl"].max(),
        "worst_trade": trade_log_df["net_pnl"].min(),
    }
    trade_summary_df = pd.DataFrame([trade_summary])

display(trade_summary_df)

## Step 6: Research And Validation Split

I make the split structural rather than optimised. The point is to show how exploratory rows can be separated from later validation rows.

In [ ]:
split_timestamp = working_table["timestamp_utc"].quantile(0.65)
working_table["sample"] = np.where(
    working_table["timestamp_utc"] <= split_timestamp,
    "research",
    "validation",
)

sample_summary = (
    working_table
    .groupby("sample", as_index=False)
    .agg(
        rows=("timestamp_utc", "size"),
        signal_rows=("entry_signal", lambda s: (s != 0).sum()),
        avg_stale_minutes=("prediction_stale_minutes", "mean"),
    )
)

display(sample_summary)

## Step 7: Visual Check

In [ ]:
plot_table = working_table.dropna(subset=["prediction_price"]).copy()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(plot_table["timestamp_utc"], plot_table["prediction_price"])
axes[0].set_title("Synthetic prediction price")

axes[1].plot(plot_table["timestamp_utc"], plot_table[z_col])
axes[1].axhline(ENTRY_Z, linestyle="--", linewidth=1)
axes[1].axhline(-ENTRY_Z, linestyle="--", linewidth=1)
axes[1].set_title("Placeholder rolling z-score")

if len(trade_log_df) > 0:
    equity = trade_log_df["net_pnl"].cumsum()
    axes[2].step(trade_log_df["exit_time"], equity, where="post")
axes[2].set_title("Toy cumulative net P&L by trade exit")

for ax in axes:
    ax.grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

## Public Version Omissions

I use synthetic event windows, placeholder signals, toy holding periods, and toy costs. I do not publish real events, market selection, thresholds, sizing rules, or performance from private research.